In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
import urllib.parse

In [2]:
def get_db_connection():
    
    user = os.getenv('MYSQL_USER')
    password = os.getenv('MYSQL_PASSWORD')
    host = os.getenv('MYSQL_HOST')
    port = os.getenv('MYSQL_PORT')
    database = os.getenv('MYSQL_DB')

    safe_password = urllib.parse.quote_plus(password)

    db_url = f"mysql+pymysql://{user}:{safe_password}@{host}:{port}/{database}"
    engine = create_engine(db_url)
    return engine

def query_data(query):
    try:
        engine = get_db_connection()
        df_result = pd.read_sql(query,con=engine)
        return df_result
    except Exception as e:
        print(f"Error in querying data: {e}")
        return None

In [3]:
household_2024_query = """
select exp.newid,exp.seqno,exp.expname,exp.cost_,exp.ref_mo,exp.ref_yr,exp.gift,exp.ucc,exp.cost,household.fam_size,household.family_income_before_tax,household.calibration_weight,household.number_of_earners,household.popsize,household.interview_month,household.interview_year,household.region,household.sex_ref,household.total_expenditure_prior_quarter,household.total_expenditure_current_quarter,household.state,household.imputed_income_before_tax,household.psu,household.division,household.urban from monthly_expenditure exp left join household_2024_metadata household on exp.newid = household.newid where exp.ucc in ('690114','690111','690120','270102','270106','690113',
     '690114','690310','300311','300312','300321','300322','300331','300332','320522','320232','690117','690119','690116',
     '480100','480213','490501','310316','310140','270310','620930','310231','310232','310400','340610','340902','310314',
     '310350','610130','310243','620917','620918','310333','690320','690330','590230','690118','300311','300312','300321',
     '300322','320331','320332','320522','320232','690111','690117','690119','690120','690115','690116','690210','270106','690310',
     '620930','270310','310140','310231','310232','620917','620918','310243','310400','310316','310314','610130','310333','310350') and exp.ref_yr = 2024;
"""

df_2024_data = query_data(household_2024_query)

In [4]:
df_2024_data.head()

,newid,seqno,expname,cost_,ref_mo,ref_yr,gift,ucc,cost,fam_size,...,interview_year,region,sex_ref,total_expenditure_prior_quarter,total_expenditure_current_quarter,state,imputed_income_before_tax,psu,division,urban
0,5348484,17,QADOTHX,E,1,2024,2,270310,3.0,2,...,2024,2.0,1,10609.3333,6501.1667,17.0,170000.0,S23A,3.0,1
1,5356724,10,QADOTHX,E,2,2024,2,270310,4.0,1,...,2024,4.0,1,4515.7500,8258.5000,15.0,91200.0,S49F,9.0,1
2,5357274,14,QADOTHX,E,2,2024,2,270310,3.0,1,...,2024,4.0,1,3666.5000,9154.0000,2.0,83311.0,S49G,9.0,1
3,5358004,25,QADOTHX,E,2,2024,2,270310,5.0,1,...,2024,3.0,2,2597.6667,12292.3333,24.0,145058.9,S35E,5.0,1
4,5360154,23,QADOTHX,E,2,2024,2,270310,8.0,1,...,2024,3.0,2,4582.0833,10772.1667,12.0,44537.8,None,5.0,1


In [10]:
#df_2024_data.drop_duplicates(inplace=True)

#df_2024_data.dropna(inplace = True)

ucc_2024_map ={
    # Telecommunications
    270102: "Cellular phone service",
    270106: "Residential telephone including VOIP",
    270310: "Cable and satellite television services",
    690114: "Computer information services (internet)",
    690116: "Internet services away from home",
    
    # Computing Hardware & Accessories
    690111: "Computers and computer hardware for nonbusiness use",
    690117: "Portable memory",
    690120: "Computer accessories",
    690115: "Personal digital assistants",
    320232: "Telephones and accessories",
    690210: "Telephone answering devices",
    
    # Software & Digital Services
    690119: "Computer software",
    620930: "Online gaming services",
    310400: "Applications, games, and ringtones for handheld devices",
    
    # Computing Services
    690113: "Repair of computer systems for nonbusiness use",
    690310: "Installation of computers",
    
    # Streaming & Digital Media
    310350: "Streaming and downloading audio",
    310243: "Rental, streaming, and downloading videos",
    620917: "Rental of video hardware/accessories",
    620918: "Rental of video software",
    
    # Gaming
    310231: "Video game software",
    310232: "Video game hardware and accessories",
    
    # Audio/Visual Equipment
    310140: "Televisions",
    310316: "Stereos, radios, speakers, and sound components",
    310314: "Personal digital audio players",
    310333: "Accessories and other sound equipment",
    340610: "Repair of televisions, radio, and sound equipment",
    340902: "Rental of televisions",
    690320: "Installation of televisions",
    690330: "Installation of satellite television equipment",
    
    # Musical Instruments
    610130: "Musical instruments and accessories",
    
    # Digital Reading
    590230: "Books, digital books, or book subscriptions",
    690118: "Digital book readers",
    
    # Appliances (Non-Digital - appear to be duplicates/errors in original list)
    300311: "Cooking stoves and ovens (renter)",
    300312: "Cooking stoves and ovens (owned home)",
    300321: "Microwave ovens (renter)",
    300322: "Microwave ovens (owned home)",
    300331: "Portable dishwashers (renter)",
    300332: "Portable dishwashers (owned home)",
    320522: "Portable heating and cooling equipment",
    
    # Vehicle Accessories (Non-Digital - appear to be errors in original list)
    480100: "Vehicle parts, accessories, fluid excluding tires",
    480213: "Parts, equipment, and accessories",
    490501: "Vehicle accessories including labor",

}

df_2024_data['product_description'] = df_2024_data['ucc'].map(ucc_2024_map).fillna(df_2024_data['ucc'])
print(df_2024_data.head())

#df_2024_data.drop(columns=['ALCNO','PUBFLAG','UCCSEQ'],inplace=True)

#df_2024_data['GIFT'] = df_2024_data['GIFT'].replace({1: True, 2: False}).astype(bool)

     newid  seqno  expname cost_  ref_mo  ref_yr  gift     ucc  cost  \
0  5348484     17  QADOTHX     E       1    2024     2  270310   3.0   
1  5356724     10  QADOTHX     E       2    2024     2  270310   4.0   
2  5357274     14  QADOTHX     E       2    2024     2  270310   3.0   
3  5358004     25  QADOTHX     E       2    2024     2  270310   5.0   
4  5360154     23  QADOTHX     E       2    2024     2  270310   8.0   

   fam_size  ...  region  sex_ref  total_expenditure_prior_quarter  \
0         2  ...     2.0        1                       10609.3333   
1         1  ...     4.0        1                        4515.7500   
2         1  ...     4.0        1                        3666.5000   
3         1  ...     3.0        2                        2597.6667   
4         1  ...     3.0        2                        4582.0833   

   total_expenditure_current_quarter  state  imputed_income_before_tax   psu  \
0                          6501.1667   17.0                   1700

In [11]:
df_2024_data.columns

Index(['newid', 'seqno', 'expname', 'cost_', 'ref_mo', 'ref_yr', 'gift', 'ucc',
       'cost', 'fam_size', 'family_income_before_tax', 'calibration_weight',
       'number_of_earners', 'popsize', 'interview_month', 'interview_year',
       'region', 'sex_ref', 'total_expenditure_prior_quarter',
       'total_expenditure_current_quarter', 'state',
       'imputed_income_before_tax', 'psu', 'division', 'urban',
       'product_description'],
      dtype='object')

In [12]:
df_2024_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14916 entries, 0 to 14915
Data columns (total 26 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   newid                              14916 non-null  int64  
 1   seqno                              14916 non-null  int64  
 2   expname                            14916 non-null  object 
 3   cost_                              14916 non-null  object 
 4   ref_mo                             14916 non-null  int64  
 5   ref_yr                             14916 non-null  int64  
 6   gift                               14916 non-null  int64  
 7   ucc                                14916 non-null  int64  
 8   cost                               14916 non-null  float64
 9   fam_size                           14916 non-null  int64  
 10  family_income_before_tax           14916 non-null  int64  
 11  calibration_weight                 14916 non-null  flo

In [13]:
df_2024_data.describe()

,newid,seqno,ref_mo,ref_yr,gift,ucc,cost,fam_size,family_income_before_tax,calibration_weight,...,interview_month,interview_year,region,sex_ref,total_expenditure_prior_quarter,total_expenditure_current_quarter,state,imputed_income_before_tax,division,urban
count,1.491600e+04,14916.000000,14916.000000,14916.0,14916.000000,14916.000000,14916.000000,14916.000000,14916.000000,14916.000000,...,14916.000000,14916.0,14588.000000,14916.000000,14916.000000,14916.000000,13649.000000,14916.000000,13853.000000,14916.000000
mean,5.493857e+06,28.188656,1.322204,2024.0,1.998458,407490.747721,88.305556,2.414119,107476.207093,29192.922495,...,2.640990,2024.0,2.734851,1.503285,10400.194891,11563.333956,27.243534,122277.814173,5.432397,1.171896
std,9.665183e+04,17.432160,0.467336,0.0,0.039239,183231.160263,170.391364,1.324186,116044.033962,12007.353886,...,0.479726,0.0,1.049093,0.500006,11432.998846,10626.875286,16.495773,117610.301431,2.566048,0.377303
min,5.343864e+06,1.000000,1.000000,2024.0,1.000000,270102.000000,1.000000,1.000000,-6083.000000,1592.845000,...,2.000000,2024.0,1.000000,1.000000,146.500000,73.250000,1.000000,-26951.900000,1.000000,1.000000
25%,5.425793e+06,17.000000,1.000000,2024.0,2.000000,270102.000000,25.000000,1.000000,33258.000000,21876.577000,...,2.000000,2024.0,2.000000,1.000000,4285.425050,5204.450000,12.000000,42485.100000,3.000000,1.000000
50%,5.553012e+06,26.000000,1.000000,2024.0,2.000000,310243.000000,61.000000,2.000000,75000.000000,29410.856000,...,3.000000,2024.0,3.000000,2.000000,7168.750000,8599.400100,27.000000,86628.000000,5.000000,1.000000
75%,5.587151e+06,36.000000,2.000000,2024.0,2.000000,690114.000000,104.000000,3.000000,142784.000000,35692.829000,...,3.000000,2024.0,4.000000,2.000000,12259.666700,13985.125000,41.000000,155000.000000,8.000000,1.000000
max,5.608061e+06,718.000000,2.000000,2024.0,2.000000,690320.000000,9000.000000,10.000000,924823.000000,91983.018000,...,3.000000,2024.0,4.000000,2.000000,204944.087500,115150.400000,55.000000,949344.200000,9.000000,2.000000


In [14]:
df_2024_data.isna().sum()

newid                                   0
seqno                                   0
expname                                 0
cost_                                   0
ref_mo                                  0
ref_yr                                  0
gift                                    0
ucc                                     0
cost                                    0
fam_size                                0
family_income_before_tax                0
calibration_weight                      0
number_of_earners                       0
popsize                                 0
interview_month                         0
interview_year                          0
region                                328
sex_ref                                 0
total_expenditure_prior_quarter         0
total_expenditure_current_quarter       0
state                                1267
imputed_income_before_tax               0
psu                                  8096
division                          

In [15]:
df_2024_data.dropna(inplace=True)

In [20]:
df_2024_data.drop_duplicates(inplace=True)

In [21]:
df_2024_data.head()

,newid,seqno,expname,cost_,ref_mo,ref_yr,gift,ucc,cost,fam_size,...,region,sex_ref,total_expenditure_prior_quarter,total_expenditure_current_quarter,state,imputed_income_before_tax,psu,division,urban,product_description
0,5348484,17,QADOTHX,E,1,2024,2,270310,3.0,2,...,2.0,1,10609.3333,6501.1667,17.0,170000.0,S23A,3.0,1,Cable and satellite television services
1,5356724,10,QADOTHX,E,2,2024,2,270310,4.0,1,...,4.0,1,4515.7500,8258.5000,15.0,91200.0,S49F,9.0,1,Cable and satellite television services
2,5357274,14,QADOTHX,E,2,2024,2,270310,3.0,1,...,4.0,1,3666.5000,9154.0000,2.0,83311.0,S49G,9.0,1,Cable and satellite television services
3,5358004,25,QADOTHX,E,2,2024,2,270310,5.0,1,...,3.0,2,2597.6667,12292.3333,24.0,145058.9,S35E,5.0,1,Cable and satellite television services
7,5366304,17,QADOTHX,E,1,2024,2,270310,10.0,3,...,3.0,2,3983.3333,8911.6667,48.0,72000.0,S37B,7.0,1,Cable and satellite television services


In [22]:
df_2024_data['gift']=df_2024_data['gift'].replace({1: True, 2: False}).astype(bool)

/tmp/ipykernel_603038/1690733136.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_2024_data['gift']=df_2024_data['gift'].replace({1: True, 2: False}).astype(bool)


In [23]:
df_2024_data.head()

,newid,seqno,expname,cost_,ref_mo,ref_yr,gift,ucc,cost,fam_size,...,region,sex_ref,total_expenditure_prior_quarter,total_expenditure_current_quarter,state,imputed_income_before_tax,psu,division,urban,product_description
0,5348484,17,QADOTHX,E,1,2024,False,270310,3.0,2,...,2.0,1,10609.3333,6501.1667,17.0,170000.0,S23A,3.0,1,Cable and satellite television services
1,5356724,10,QADOTHX,E,2,2024,False,270310,4.0,1,...,4.0,1,4515.7500,8258.5000,15.0,91200.0,S49F,9.0,1,Cable and satellite television services
2,5357274,14,QADOTHX,E,2,2024,False,270310,3.0,1,...,4.0,1,3666.5000,9154.0000,2.0,83311.0,S49G,9.0,1,Cable and satellite television services
3,5358004,25,QADOTHX,E,2,2024,False,270310,5.0,1,...,3.0,2,2597.6667,12292.3333,24.0,145058.9,S35E,5.0,1,Cable and satellite television services
7,5366304,17,QADOTHX,E,1,2024,False,270310,10.0,3,...,3.0,2,3983.3333,8911.6667,48.0,72000.0,S37B,7.0,1,Cable and satellite television services
